# Chapter 17 — Bias, Variance, and Regularization

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 4 (`code/ch04/gen_retail.py` in the repository).

In [2]:
import numpy as np, pandas as pd

rng = np.random.default_rng(7)
items = [("Cold Brew Concentrate", 12.50, "Beverage"),
         ("Ceramic Mug", 8.75, "Merch"),
         ("Espresso Beans 1kg", 24.00, "Beans"),
         ("Paper Filters x100", 4.25, "Supplies"),
         ("Travel Tumbler", 18.90, "Merch"),
         ("Decaf Beans 1kg", 22.50, "Beans"),
         ("Milk Frother", 31.00, "Equipment"),
         ("Gift Card", 25.00, "Other")]
ctry = ["United Kingdom", "Germany", "France",
        "Netherlands", "Ireland"]

rows, inv = [], 536000
for month in range(1, 13):
    for _ in range(int(rng.integers(55, 85))):
        inv += 1
        c = str(rng.choice(ctry, p=[.55, .15, .12, .10, .08]))
        cust = int(rng.integers(12000, 12400))
        day = int(rng.integers(1, 29))
        for _ in range(int(rng.integers(1, 4))):
            i = int(rng.integers(0, len(items)))
            rows.append({"InvoiceNo": str(inv),
                "StockCode": "S%d" % (1000 + i),
                "Description": items[i][0], "Category": items[i][2],
                "Quantity": int(rng.integers(1, 13)),
                "InvoiceDate": "2024-%02d-%02d" % (month, day),
                "UnitPrice": items[i][1], "CustomerID": cust,
                "Country": c})

df = pd.DataFrame(rows)
cancel = df.sample(18, random_state=3).index
df.loc[cancel, "InvoiceNo"] = "C" + df.loc[cancel, "InvoiceNo"]
df.loc[cancel, "Quantity"] = -df.loc[cancel, "Quantity"]
df.loc[df.sample(40, random_state=5).index, "CustomerID"] = np.nan
df.to_csv("retail.csv", index=False)
print(f"wrote retail.csv: {len(df):,} rows, {df.InvoiceNo.nunique():,} invoices")

wrote retail.csv: 1,803 rows, 914 invoices


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch17/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
rng = np.random.default_rng(0)
# retail.csv is created by Chapter 4 (code/ch04/gen_retail.py). The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("retail.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "retail.csv")):
            _shutil.copy(_os.path.join(_d, "retail.csv"), "retail.csv"); break

## The chapter code

### Block 1  (`c1.py`)

In [4]:
import numpy as np
rng = np.random.default_rng(0)

def truth(x):                       # the relationship, unknown to any model
    return np.sin(1.4 * x) + 0.3 * x

def sample(n=40):
    x = rng.uniform(-3, 3, n)
    return x, truth(x) + rng.normal(0, 0.45, n)

xs = np.linspace(-3, 3, 200)        # where we measure
runs = 300                          # 300 alternative worlds
NOISE = 0.45 ** 2

print(f"{'degree':>7}{'bias^2':>9}{'variance':>10}{'noise':>8}{'total':>10}")
for deg in [1, 2, 3, 5, 7, 9, 12]:
    preds = np.zeros((runs, len(xs)))
    for r in range(runs):
        x, y = sample()
        preds[r] = np.polyval(np.polyfit(x, y, deg), xs)
    bias2 = ((preds.mean(0) - truth(xs)) ** 2).mean()
    var = preds.var(0).mean()
    print(f"{deg:>7}{bias2:>9.3f}{var:>10.3f}{NOISE:>8.3f}"
          f"{bias2 + var + NOISE:>10.3f}")

 degree   bias^2  variance   noise     total
      1    0.440     0.038   0.203     0.680
      2    0.441     0.065   0.203     0.708
      3    0.045     0.037   0.203     0.284
      5    0.001     0.052   0.203     0.256
      7    0.001     0.264   0.203     0.468
      9    0.037     5.796   0.203     6.035
     12    0.741   623.162   0.203   624.106


### Block 2  (`c2.py`)

In [5]:
rng = np.random.default_rng(0)
# The two diagnostics that tell you which problem you have.
X = rng.uniform(-3, 3, size=(30, 1))
y = (np.sin(1.4 * X[:, 0]) + 0.3 * X[:, 0]
     + rng.normal(0, 0.45, 30))
cv = KFold(5, shuffle=True, random_state=0)

print(f"{'degree':>7}{'train MSE':>11}{'val MSE':>10}   diagnosis")
for deg in [1, 3, 5, 9, 14]:
    m = make_pipeline(PolynomialFeatures(deg), StandardScaler(),
                      LinearRegression())
    val = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(X, y)
    tr = ((y - m.predict(X)) ** 2).mean()
    d = ("underfit (high bias)" if tr > 0.35 else
         "overfit (high variance)" if val > 2 * tr + 0.1 else "balanced")
    print(f"{deg:>7}{tr:>11.3f}{val:>10.3f}   {d}")

 degree  train MSE   val MSE   diagnosis
      1      0.433     0.545   underfit (high bias)
      3      0.117     0.175   balanced
      5      0.093     0.147   balanced
      9      0.090     0.254   balanced
     14      0.071    75.852   overfit (high variance)


### Block 3  (`c3.py`)

In [6]:
rng = np.random.default_rng(0)
# The same overfit model, rescued by a penalty instead of less capacity.
X = rng.uniform(-3, 3, size=(30, 1))
y = np.sin(1.4 * X[:, 0]) + 0.3 * X[:, 0] + rng.normal(0, 0.45, 30)
cv = KFold(5, shuffle=True, random_state=0)

print(f"{'alpha':>9}{'train MSE':>11}{'val MSE':>10}{'largest |w|':>13}")
for a in [0.0, 1e-4, 1e-2, 1e-1, 1.0, 10.0, 100.0]:
    m = make_pipeline(PolynomialFeatures(14), StandardScaler(),
                      Ridge(alpha=a) if a else LinearRegression())
    val = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(X, y)
    tr = ((y - m.predict(X)) ** 2).mean()
    w = np.abs(m[-1].coef_).max()
    print(f"{a:>9.4f}{tr:>11.3f}{val:>10.3f}{w:>13.1f}")

    alpha  train MSE   val MSE  largest |w|
   0.0000      0.071    75.852       1963.5
   0.0001      0.089     0.271          8.7
   0.0100      0.092     0.176          1.9
   0.1000      0.095     0.154          1.7
   1.0000      0.126     0.198          1.2
  10.0000      0.287     0.409          0.5
 100.0000      0.560     0.665          0.1


### Block 4  (`c4.py`)

In [7]:
rng = np.random.default_rng(0)
import pandas as pd
df = pd.read_csv("retail.csv")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
clean = df[~df["InvoiceNo"].str.startswith("C")].copy()

o = clean.groupby("InvoiceNo").agg(
        rev=("Revenue", "sum"), units=("Quantity", "sum"),
        lines=("StockCode", "count"),
        avg_price=("UnitPrice", "mean")).reset_index()

# Two features that genuinely matter, one near-duplicate of units,
# and six columns of pure noise -- the shape of a real feature table.
o["units_copy"] = o["units"] * 1.02 + rng.normal(0, .3, len(o))
for j in range(6):
    o[f"noise_{j}"] = rng.normal(0, 1, len(o))

feats = ["units", "lines", "avg_price", "units_copy"] + \
        [f"noise_{j}" for j in range(6)]
print(f"{len(o):,} orders, {len(feats)} candidate features")
print(f"correlation units vs units_copy: "
      f"{o['units'].corr(o['units_copy']):.3f}")

897 orders, 10 candidate features
correlation units vs units_copy: 0.999


### Block 5  (`c5.py`)

In [8]:
Xo, yo = o[feats].values, o["rev"].values
cv = KFold(5, shuffle=True, random_state=0)

def run(model):
    m = make_pipeline(StandardScaler(), model)
    mse = -cross_val_score(m, Xo, yo, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(Xo, yo)
    return mse, m[-1].coef_

print(f"{'model':<18}{'CV MSE':>10}{'non-zero':>10}")
for name, mdl in [("plain", LinearRegression()),
                  ("ridge  (a=10)", Ridge(alpha=10)),
                  ("lasso  (a=1)", Lasso(alpha=1.0, max_iter=50000)),
                  ("lasso  (a=5)", Lasso(alpha=5.0, max_iter=50000))]:
    mse, w = run(mdl)
    print(f"{name:<18}{mse:>10.1f}{int((np.abs(w) > 1e-6).sum()):>10}")

print()
_, w_ridge = run(Ridge(alpha=10))
_, w_lasso = run(Lasso(alpha=5.0, max_iter=50000))
print(f"{'feature':<12}{'ridge':>9}{'lasso':>9}")
for f, a, b in zip(feats, w_ridge, w_lasso):
    print(f"{f:<12}{a:>9.1f}{b:>9.1f}")

model                 CV MSE  non-zero
plain                 3021.7        10
ridge  (a=10)         3039.9        10
lasso  (a=1)          2999.0         4
lasso  (a=5)          3035.3         2

feature         ridge    lasso
units            73.5    126.7
lines            -0.4      0.0
avg_price        72.2     67.8
units_copy       57.8      0.0
noise_0          -0.9     -0.0
noise_1          -0.1     -0.0
noise_2           0.7      0.0
noise_3           0.4      0.0
noise_4          -1.0     -0.0
noise_5          -2.9     -0.0
